# Preface - Group to Species

In 1986's article, the calculation was done in the global level - all fish were assumed to be on TL=2, with TE=0.1.  
In 1995's article the calculation was better, with mean TL for each commercial group, where fractional TL was calculated for each species and then  
species with similar fractional TL were aggregated to the same commercial group. TE was still assumed to be 0.1 globally.

In this notebook we'd check the effect of dis-aggeregating 1995's PPR calculation method to the level of species:
1. still using global TE=0.1
2. still using fractional TL for each species

Since it seems like the "species group" used in 1995's article are not either commercial or functional groups,  
we will perform the analysis on a sample catch area (the Baltic Sea), and see how converting groups to species can increase PPR.

In [194]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Databases

We are using:  
1. "2020 sup.xlsx" - [2020's article]((https://www-sciencedirect-com.ezproxy.weizmann.ac.il/science/article/pii/S0048969720308627?source=WISbookmarklet#s0085)), [supplementary 2](https://ars-els-cdn-com.ezproxy.weizmann.ac.il/content/image/1-s2.0-S0048969720308627-mmc2.xlsx).  
    mapping scientific names of species to their fractional TL.
2. "SAU LME 26 v50-1.csv" - Catches by Taxon in the waters of Mediterranean Sea, Downloaded form [SeeAroundUs](https://www.seaaroundus.org/data/#/lme/26?chart=catch-chart&dimension=taxon&measure=tonnage&limit=10&sciname=false).  
    mapping scientific names to functional and commercial groups, and to tonnes catch.

In [195]:
# read dfs:
df_TLs_orig = pd.read_excel("2020 sup.xlsx", sheet_name="SPPR of species in database ")
df_catch_orig = pd.read_csv("SAU LME 26 v50-1.csv")  # Mediterranian Sea
# df_catch_orig = pd.read_csv("SAU LME 23 v50-1.csv")  # Baltic Sea
# df_catch_orig = pd.read_csv("SAU LME 13 v50-1.csv")  # Humboldt current
# df_catch_orig = pd.read_csv("SAU LME 46 v50-1.csv")  # new Zealand shelf


### Process df_catch:

In [196]:
# take a look at the data:
df_catch_orig.head()

,area_name,area_type,year,scientific_name,common_name,functional_group,commercial_group,fishing_entity,fishing_sector,catch_type,reporting_status,gear_type,end_use_type,tonnes,landed_value
0,Mediterranean Sea,lme,1950,Miscellaneous diadromous fishes,Diadromous fishes nei,Large demersals (>=90 cm),Other fishes & inverts,Turkey,Subsistence,Landings,Unreported,subsistence fishing gear,Direct human consumption,1004.029783,82.706818
1,Mediterranean Sea,lme,1950,Miscellaneous diadromous fishes,Diadromous fishes nei,Large demersals (>=90 cm),Other fishes & inverts,Turkey,Recreational,Landings,Unreported,recreational fishing gear,Direct human consumption,111.558865,9.189646
2,Mediterranean Sea,lme,1950,Marine fishes not identified,Marine fishes nei,Medium demersals (30 - 89 cm),Other fishes & inverts,Albania,Industrial,Discards,Unreported,bottom trawl,NaN,132.791079,NaN
3,Mediterranean Sea,lme,1950,Marine fishes not identified,Marine fishes nei,Medium demersals (30 - 89 cm),Other fishes & inverts,Albania,Industrial,Landings,Reported,bottom trawl,Direct human consumption,680.298912,139659.400792
4,Mediterranean Sea,lme,1950,Marine fishes not identified,Marine fishes nei,Medium demersals (30 - 89 cm),Other fishes & inverts,Albania,Industrial,Landings,Reported,bottom trawl,Other,0.680980,0.018178


In [197]:
df_catch_orig['catch_type'].value_counts(dropna=False)

catch_type
Landings    1008925
Discards     136681
Name: count, dtype: int64

In [198]:
df_catch = df_catch_orig.copy()

# choose columns to work with; use data from all fishing entities, all fishing sectors, all reporting status, all gear types, and all end_use_types.
df_catch = df_catch.reindex(columns=[
    'scientific_name',
    'common_name',
    'functional_group',
    'commercial_group',
    'tonnes',
    'catch_type',
    'year',
])

# filter year in [1988, 1991]:
catch_before_filter = df_catch['tonnes'].sum()
df_catch = df_catch[(1988 <= df_catch['year']) & (df_catch['year'] <= 1991)]
print(f'catch in 1988-1991 / Total catch: {int(100*round(df_catch['tonnes'].sum()/catch_before_filter, 2))}%')

# filter "Discards" catch types:
catch_before_filter = df_catch['tonnes'].sum()
df_catch = df_catch[df_catch['catch_type'] == 'Landings']
print(f'Landings / Total Catch 1988-1991: {int(100*round(df_catch['tonnes'].sum()/catch_before_filter, 2))}%')

# drop used columns:
df_catch.drop(columns=['catch_type', 'year'], inplace=True)

# group and aggregate by scientific name:
df_catch = df_catch.groupby('scientific_name').agg({
    'common_name' : 'first',
    'functional_group' : 'first',
    'commercial_group' : 'first',
    'tonnes' : 'sum',
}).reset_index()

# take a look again:
print(f'scientific names are unique: {len(df_catch) == len(set(df_catch['scientific_name']))}')
df_catch.head()

catch in 1988-1991 / Total catch: 7%
Landings / Total Catch 1988-1991: 90%
scientific names are unique: True


,scientific_name,common_name,functional_group,commercial_group,tonnes
0,Acanthocardia aculeata,Spiny cockle,Other demersal invertebrates,Molluscs,2989.741843
1,Acanthocardia echinata,European prickly cockle,Other demersal invertebrates,Molluscs,3013.360685
2,Acanthocardia tuberculata,Tuberculate cockle,Other demersal invertebrates,Molluscs,8824.007831
3,Acanthocybium solandri,Wahoo,Large pelagics (>=90 cm),Perch-likes,9.300116
4,Aequipecten opercularis,Queen scallop,Other demersal invertebrates,Molluscs,10.692382


### Process df_TLs:

In [199]:
df_TLs_orig.head()

,No,Scientific name,Habitat,TLtype,MeanTL,Averaged SPPRregression\n(tonnes-NPP/tonne-fish in wet weight),95% prediction interval,Unnamed: 7,SPPRclassical\n(tonnes-NPP/tonne-fish in wet weight),Method for estimation \nof Averaged SPPRs
0,NaN,NaN,NaN,NaN,NaN,NaN,lwr,upr,NaN,NaN
1,1.0,Abalistes stellaris,demersal,DietTroph,3.64,753.207488,206.30658,2749.895428,275.775717,Interpolation
2,2.0,Ablennes hians,reef-associated,FoodTroph,4.50,9453.915355,2481.748278,36013.529791,1720.226506,Extrapolation
3,3.0,Abludomelita obtusata,benthic,FoodTroph,2.00,6.049575,1.453074,25.186154,8.403361,Extrapolation
4,4.0,Abralia astrosticta,pelagic,FoodTroph,3.83,1317.194870,359.813355,4821.950886,413.239759,Interpolation


In [200]:
df_TLs = df_TLs_orig.copy()

# choose columns to work with; use data from all TLtypes.
df_TLs = df_TLs.reindex(columns=[
    'Scientific name',
    'Habitat',
    'MeanTL',
])

# rename so that naming matches df_catch's naming scheme:
df_TLs = df_TLs.rename(columns={
    'Scientific name': 'scientific_name',
    'Habitat': 'habitat',
    'MeanTL': 'mean_TL'
})

# filter rows where scientific_name is missing:
df_TLs = df_TLs.dropna(subset=['scientific_name'])

# take a look again:
print(f'scientific names are unique: {len(df_TLs) == len(set(df_TLs['scientific_name']))}')  # False

# seems like scientific_name is not unique; run the following lines to see those rows:
# df_TLs['scientific_name'].value_counts(dropna=False)
# df_TLs[df_TLs.duplicated(subset=['scientific_name'], keep=False)]

# groupby scientific name and aggregate; all duplicates are from the same habitat, so we can use regular mean to get mean_TL:
df_TLs = df_TLs.groupby('scientific_name').agg({
    'habitat' : 'first',
    'mean_TL' : 'mean',
}).reset_index()

# take a look again:
print(f'scientific names are unique: {len(df_TLs) == len(set(df_TLs['scientific_name']))}')  # True
df_TLs.head()

scientific names are unique: False
scientific names are unique: True


,scientific_name,habitat,mean_TL
0,Abalistes stellaris,demersal,3.64
1,Ablennes hians,reef-associated,4.50
2,Abludomelita obtusata,benthic,2.00
3,Abralia astrosticta,pelagic,3.83
4,Abralia heminuchalis,pelagic,3.83


In [201]:
# [len(x.split(' ')) < 2 for x in df_TLs['scientific_name'].values]

# df_TLs[[len(x.split(' ')) < 2 for x in df_TLs['scientific_name'].values]]

## Merge dfs:

In [202]:
def average_TL(df, TE=0.1):
    df = df[['mean_TL', 'tonnes']].dropna(subset='mean_TL')
    tls = df['mean_TL'].values
    tonnes = df['tonnes'].values
    if len(tonnes) > 0:
        # a = 1 + np.log(np.average(TE ** (tls-1), weights=tonnes))/np.log(TE)
        # a = 1 + np.log(np.average(TE ** (tls-1)))/np.log(TE)
        # a = np.mean(tls)
        a = np.average(tls, weights=tonnes)
        return a
    else:
        return np.nan

def fillna_mean_TL(df, by='commercial_group'):
    tl_map = df.groupby(by).apply(average_TL, include_groups=False)
    is_null = df['mean_TL'].isnull()
    df.loc[is_null, 'mean_TL'] = df.loc[is_null, by].map(tl_map)
    return df

def print_nan_TL_info(df):
    catch_before_filter = df_catch['tonnes'].sum()
    df = df.dropna(subset=['mean_TL'])
    print(f'Species with non-null TL / Total Species: {len(df['scientific_name'])} / {len(df_catch['scientific_name'])} = {int(100*round(len(df['scientific_name'])/len(df_catch['scientific_name']), 2))}%')  # 53%
    print(f'Catch of non-null TL / Total Catch: {round(df['tonnes'].sum())} / {round(catch_before_filter)} = {int(100*round(df['tonnes'].sum()/catch_before_filter, 2))}%')
    print()

In [219]:
df_catch['scientific_name'] = df_catch['scientific_name'].str.strip()
df_TLs['scientific_name'] = df_TLs['scientific_name'].str.strip()

# merge on scientific_name:
df_merged = pd.merge(
    left=df_catch,
    right=df_TLs,
    how='left',
    on='scientific_name'
)

# filter rows where mean_TL is missing:
print_nan_TL_info(df_merged)
# This is too much! we need to try to complete the data on TLs.

# fill nan values by first averaging (spacial average) on genus and then on commercial group:
df_merged['genus'] = df_merged['scientific_name'].apply(lambda x: x.split(' ')[0])
df = fillna_mean_TL(df_merged, by='genus')
# df = fillna_mean_TL(df, by='commercial_group')

print_nan_TL_info(df)

df_complete = df.copy()
df_complete

Species with non-null TL / Total Species: 177 / 344 = 51%
Catch of non-null TL / Total Catch: 4785393 / 7615203 = 63%

Species with non-null TL / Total Species: 217 / 344 = 63%
Catch of non-null TL / Total Catch: 5547267 / 7615203 = 73%



,scientific_name,common_name,functional_group,commercial_group,tonnes,habitat,mean_TL,genus
0,Acanthocardia aculeata,Spiny cockle,Other demersal invertebrates,Molluscs,2989.741843,NaN,NaN,Acanthocardia
1,Acanthocardia echinata,European prickly cockle,Other demersal invertebrates,Molluscs,3013.360685,NaN,NaN,Acanthocardia
2,Acanthocardia tuberculata,Tuberculate cockle,Other demersal invertebrates,Molluscs,8824.007831,NaN,NaN,Acanthocardia
3,Acanthocybium solandri,Wahoo,Large pelagics (>=90 cm),Perch-likes,9.300116,pelagic-oceanic,4.26,Acanthocybium
4,Aequipecten opercularis,Queen scallop,Other demersal invertebrates,Molluscs,10.692382,benthic,2.00,Aequipecten
...,...,...,...,...,...,...,...,...
339,Venus verrucosa,Warty venus,Other demersal invertebrates,Molluscs,7.162472,NaN,NaN,Venus
340,Xiphias gladius,Swordfish,Large pelagics (>=90 cm),Tuna & billfishes,62861.944403,pelagic-oceanic,4.53,Xiphias
341,Xiphiidae,Swordfishes,Large pelagics (>=90 cm),Tuna & billfishes,65.055038,NaN,NaN,Xiphiidae
342,Zeidae,Dories,Medium bathydemersals (30 - 89 cm),Other fishes & inverts,0.152153,NaN,NaN,Zeidae


In [204]:
# bins = np.linspace(df['mean_TL'].min(), df['mean_TL'].max(), 20)
#
# df['mean_TL'].hist(by=df['commercial_group'], figsize=(15, 10), sharex=True, bins=bins)
# # df['mean_TL'].hist(by=df['functional_group'], figsize=(15, 10), sharex=True, bins=bins)
#
# plt.suptitle('Histogram of mean_TL grouped by commercial_group')
# plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
# plt.show()

## try to complete data from fishbase:

In [205]:
# import pandas as pd
# import requests
# import re
# from io import StringIO
# 
# # URLs for mediterranian sea:
# # species_url = "https://www.fishbase.se/TrophicEco/FishEcoList.php?ve_code=13"
# # resilieance_url = "https://www.fishbase.se/TrophicEco/ResilienceFishList.php?ve_code=13"
# # ecology_mat_url = "https://www.fishbase.se/report/KeyFactsMatrixList.php?e_code=13"
# 
# # URLs for Baltic sea:
# species_url = "https://www.fishbase.se/TrophicEco/FishEcoList.php?ve_code=104"
# resilieance_url = "https://www.fishbase.se/TrophicEco/ResilienceFishList.php?ve_code=104"
# ecology_mat_url = "https://www.fishbase.se/report/KeyFactsMatrixList.php?e_code=104"
# 
# # URLs for Humboldt Current:
# # species_url = "https://www.fishbase.se/TrophicEco/FishEcoList.php?ve_code=237"
# # resilieance_url = "https://www.fishbase.se/TrophicEco/ResilienceFishList.php?ve_code=237"
# # ecology_mat_url = "https://www.fishbase.se/report/KeyFactsMatrixList.php?e_code=237"
# 
# # URLs for New zealand Shelf:
# # species_url = "https://www.fishbase.se/TrophicEco/FishEcoList.php?ve_code=164"
# # resilieance_url = "https://www.fishbase.se/TrophicEco/ResilienceFishList.php?ve_code=164"
# # ecology_mat_url = "https://www.fishbase.se/report/KeyFactsMatrixList.php?e_code=164"
# 
# 
# def get_df_from_url(url):
#     # Set a User-Agent header to mimic a web browser
#     headers = {
#         "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
#     }
# 
#     # Fetch the page content
#     response = requests.get(url, headers=headers)
# 
#     # pd.read_html returns a list of all tables found on the page
#     tables = pd.read_html(StringIO(response.text))
# 
#     # Typically, the main data table is the first or second one
#     # You can check the length and preview the tables to find the right one
#     df = tables[0]
# 
#     return df
# 
# def get_TL_from_str(text: str):
#     if not isinstance(text, str):
#         return
#     matches = re.findall(r'[+-]?\d+\.?\d*', text)
#     if matches:
#         return matches[0]
# 
# def get_TL_se_from_str(text: str):
#     if not isinstance(text, str):
#         return
#     matches = re.findall(r'[+-]?\d+\.?\d*', text)
#     if len(matches) > 1:
#         return matches[1]

In [206]:
# df_ecology_mat = get_df_from_url(ecology_mat_url)
# 
# # df_ecology_mat.head()
# 
# # Display the first few rows
# df_ecology_mat['mean_TL'] = df_ecology_mat['Trophic level'].apply(get_TL_from_str)
# df_ecology_mat['se_TL'] = df_ecology_mat['Trophic level'].apply(get_TL_se_from_str)
# 
# df_ecology_mat = df_ecology_mat[[
#     'Scientific Name',
#     'Family',
#     'Trophic level',
#     'mean_TL',
#     'se_TL'
# ]]
# df_ecology_mat = df_ecology_mat.rename(columns={
#     'Scientific Name':'scientific_name',
#     'Family':'family',
# })
# 
# df_ecology_mat['scientific_name'] = df_ecology_mat['scientific_name'].str.replace('Life history tool', '').str.strip()
# 
# print(f'scientific names are unique: {len(df_ecology_mat) == len(set(df_ecology_mat['scientific_name']))}')  # True
# df_ecology_mat

In [207]:
# df_resilieance = get_df_from_url(resilieance_url)
# df_resilieance = df_resilieance[[
#     'Species',
#     'Family',
#     'Trophic level',
# ]]
# df_resilieance = df_resilieance.rename(columns={
#     'Species':'scientific_name',
#     'Family':'family',
#     'Trophic level': 'mean_TL'
# })
# 
# df_resilieance['scientific_name'] = df_resilieance['scientific_name'].str.strip()
# 
# print(f'scientific names are unique: {len(df_resilieance) == len(set(df_resilieance['scientific_name']))}')  # True
# 
# # merge:
# df_merged = pd.merge(
#     left=df_ecology_mat,
#     right=df_resilieance,
#     how='outer',
#     on='scientific_name',
#     suffixes=('_mat', '_res')
# )

In [208]:
# df_species = get_df_from_url(species_url)
# 
# df_species = df_species[[
#     'Species',
#     'Name',
#     'Family',
#     'Habitat',
#     'Trophic Level',
# ]]
# df_species
# df_species = df_species.rename(columns={
#     'Species':'scientific_name',
#     'Name': 'common_name',
#     'Family':'family_species',
#     'Habitat': 'habitat',
#     'Trophic Level': 'mean_TL_species'
# })
# 
# df_species['scientific_name'] = df_species['scientific_name'].str.strip()
# df_species['common_name'] = df_species['common_name'].str.strip()
# 
# print(f'scientific names are unique: {len(df_species) == len(set(df_species['scientific_name']))}')  # True
# 
# # merge:
# df_merged = pd.merge(
#     left=df_merged,
#     right=df_species,
#     how='outer',
#     on='scientific_name',
#     suffixes=('_merged', '_spcies')
# )
# 
# df_merged['mean_TL'] = df_merged['mean_TL_res'].fillna(df_merged['mean_TL_mat']).fillna(df_merged['mean_TL_species'])
# df_merged['family'] = df_merged['family_species'].fillna(df_merged['family_mat']).fillna(df_merged['family_res'])
# 
# df_merged = df_merged[[
#     'scientific_name',
#     'common_name',
#     'family',
#     'habitat',
#     'mean_TL',
#     'se_TL',
# ]]
# 
# df_merged

In [209]:
# df_urls = df_merged.copy()
# 
# df_urls['scientific_name'] = df_urls['scientific_name'].str.strip()
# df_TLs['scientific_name'] = df_TLs['scientific_name'].str.strip()
# 
# df_TLs_complete = pd.merge(
#     left=df_merged,
#     right=df_TLs,
#     how='outer',
#     on='scientific_name',
#     suffixes=('_l', '_r')
# )
# duplicated_columns = [c[:-2] for c in df_TLs_complete.columns if c.endswith('_l')]
# duplicated_columns
# # for c in duplicated_columns:
# #     df_TLs_complete[c] = df_TLs_complete[c + '_l'].fillna(df_TLs_complete[c + '_r'])
# 
# # df_TLs_complete

In [210]:
# df_urls = df_merged.copy()
# df_urls

#### back to merge:

In [211]:
# # merge on scientific_name:
# df_catch['scientific_name'] = df_catch['scientific_name'].str.strip()
# 
# df = pd.merge(
#     left=df_catch,
#     right=df_urls,
#     how='left',
#     on='scientific_name',
#     suffixes=('', '_urls')
# )
# 
# df = pd.merge(
#     left=df,
#     right=df_TLs,
#     how='left',
#     on='scientific_name',
#     suffixes=('', '_2020')
# )
# 
# # fill nans:
# df['mean_TL'] = df['mean_TL'].fillna(df['mean_TL_2020'])
# df['habitat'] = df['habitat'].fillna(df['habitat_2020']).str.strip()
# df['common_name'] = df['common_name'].fillna(df['common_name_urls']).str.strip()
# 
# # reindex:
# df = df.reindex(columns=[
#     'scientific_name',
#     'common_name',
#     'family',
#     'habitat',
#     'commercial_group',
#     'functional_group',
#     'mean_TL',
#     'se_TL',
#     'tonnes',
# ])
# 
# # df[df['mean_TL'].isnull()]
# # filter rows where mean_TL is missing:
# df = df[df['scientific_name'] != 'Marine fishes not identified']
# catch_before_filter = df_catch[df_catch['scientific_name'] != 'Marine fishes not identified']['tonnes'].sum()
# df_complete = df.dropna(subset=['mean_TL', 'tonnes'])
# print(f'Catch of non-null TL / Total Catch: {round(df_complete['tonnes'].sum())} / {round(catch_before_filter)} = {int(100*round(df_complete['tonnes'].sum()/catch_before_filter, 2))}%')  # 63%
# print(f'Species with non-null TL / Total Species: {len(df_complete['scientific_name'])} / {len(df_catch['scientific_name'])} = {int(100*round(len(df_complete['scientific_name'])/len(df_catch['scientific_name']), 2))}%')  # 53%
# 
# # This is still too much! we need to try to complete the data on TLs.
# df_complete.sort_values(by='tonnes', ascending=False)

# Assumptions

1. mean TLs of a commercial group are calculated with weighted mean, where the weights are the catch of each species.
2. 

# Calculations

In [244]:
import numpy as np

# Define global constants:
TE = 0.1  # global TE, also possible to use 0.119 from 2020's article.

# merge on scientific_name:
df_merged = pd.merge(
    left=df_catch,
    right=df_TLs,
    how='left',
    on='scientific_name'
)

df_merged['genus'] = df_merged['scientific_name'].apply(lambda x: x.split(' ')[0])

### PPR by commercial groups:

In [253]:
df = df_merged.copy()

df_complete = fillna_mean_TL(df, by='genus')
df_complete = fillna_mean_TL(df_complete, by='commercial_group')

print_nan_TL_info(df_complete)

# compute PPR over groups:
df_commercial_groups = df_complete.groupby('commercial_group').agg({
    'tonnes': 'sum',
    'mean_TL': lambda x: np.average(x.astype(float), weights=df_complete.loc[x.index, 'tonnes'].astype(float)),
}).reset_index()

df_commercial_groups['SPPR'] = df_commercial_groups['mean_TL'].apply(lambda x: (1/TE)**(x-1))
df_commercial_groups['PPR'] = df_commercial_groups['tonnes'] * df_commercial_groups['SPPR']
df_commercial_groups = df_commercial_groups.sort_values(by='PPR', ascending=False)

ppr_commercial = df_commercial_groups['PPR'].sum()
print(f'commercial groups: PPR = {round(ppr_commercial/1_000_000, 2)} x 10^6 Tonnes')

Species with non-null TL / Total Species: 344 / 344 = 100%
Catch of non-null TL / Total Catch: 7615203 / 7615203 = 100%

commercial groups: PPR = 4521.71 x 10^6 Tonnes


### PPR by functional groups:

In [257]:
df = df_merged.copy()

df_complete = fillna_mean_TL(df, by='genus')
df_complete = fillna_mean_TL(df_complete, by='commercial_group')

print_nan_TL_info(df_complete)

# compute PPR over groups:
df_fucntional_groups = df_complete.groupby('functional_group').agg({
    'tonnes': 'sum',
    'mean_TL': lambda x: np.average(x.astype(float), weights=df_complete.loc[x.index, 'tonnes'].astype(float)),
}).reset_index()

df_fucntional_groups['SPPR'] = df_fucntional_groups['mean_TL'].apply(lambda x: (1/TE)**(x-1))
df_fucntional_groups['PPR'] = df_fucntional_groups['tonnes'] * df_fucntional_groups['SPPR']
df_fucntional_groups = df_fucntional_groups.sort_values(by='PPR', ascending=False)

ppr_fucntional = df_fucntional_groups['PPR'].sum()
print(f'fucntional groups: PPR = {round(ppr_fucntional/1_000_000, 2)} x 10^6 Tonnes')

Species with non-null TL / Total Species: 344 / 344 = 100%
Catch of non-null TL / Total Catch: 7615203 / 7615203 = 100%

fucntional groups: PPR = 4363.6 x 10^6 Tonnes


### PPR by species:

In [255]:
df = df_merged.copy()

df_complete = fillna_mean_TL(df, by='genus')
df_complete = fillna_mean_TL(df_complete, by='commercial_group')

df_species = df_complete.copy()

df_species['mean_TL'] = df_species['mean_TL'].astype(float)
df_species['SPPR'] = df_species['mean_TL'].apply(lambda x: (1/TE)**(x-1))
df_species['PPR'] = df_species['tonnes'] * df_species['SPPR']
df_species = df_species.sort_values(by='PPR', ascending=False)

df_species = df_species[[
    'scientific_name',
    'common_name',
    'functional_group',
    'commercial_group',
    'habitat',
    'tonnes',
    'mean_TL',
    'SPPR',
    'PPR',
]]

ppr_species = df_species['PPR'].sum()
print(f'species: PPR = {round(ppr_species/1_000_000, 2)} x 10^6 Tonnes')

species: PPR = 5595.33 x 10^6 Tonnes


# Save calculations:

In [216]:
# file_name = 'HumboldtCarrent_88-91_group2species.xlsx'
# with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
#     df_species.to_excel(writer, sheet_name='Species', index=False)
#     df_commercial_groups.to_excel(writer, sheet_name='Commercial Groups', index=False)
#     df_fucntional_groups.to_excel(writer, sheet_name='Functional Groups', index=False)